## Complex Types:
- Complex Types are `data types` is used to `store multiple values of same data type or structured fields inside a single DataFrame column`. 
- Complex Types such as `ArrayType, MapType, and StructType`.

### A. ArrayType:

It is used to `stores multiple values of the same data type in one column`.

| Employee | Skills                    |
|----------|---------------------------|
| Arun     | ["Python", "SQL", "Spark"] |
| Bala     | ["Java", "SQL"]            |


### B. MapType:

It is used to `stores key-value pairs in one column`.

| Product | Prices                           |
|---------|----------------------------------|
| Mobile  | {"Mobile": 20000, "Charger": 800} |
| Laptop  | {"Laptop": 50000, "Mouse": 800}   |


### C. StructType:

It is used to `stores name fields where one column contains different data types`.

| Employee | Address                         |
|----------|---------------------------------|
| Arun     | {city: "Chennai", pincode: 600001} |
| Bala     | {city: "Madurai", pincode: 625001} |

### A. ArrayType — employee skills

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("ArrayExample").getOrCreate()

employee_df = spark.createDataFrame([
    (101, "Arun", ["Python", "SQL", "Spark"]),
    (102, "Bala", ["Java", "SQL"])
], ["EmployeeID", "Name", "Skills"])

employee_df.show(truncate=False)

# Access the first skill
employee_df.select(
    "Name",
    col("Skills")[0].alias("FirstSkill")
).show()

## B. MapType — product prices

In [0]:
from pyspark.sql.functions import col

product_df = spark.createDataFrame([
    (101, {"Laptop": 50000, "Mouse": 800}),
    (102, {"Phone": 25000, "Charger": 1000})
], ["OrderID", "Prices"])

product_df.show(truncate=False)

# Retrieve the Laptop price
product_df.select(
    "OrderID",
    col("Prices")["Laptop"].alias("LaptopPrice")
).show()

## C. StructType — employee address

In [0]:
from pyspark.sql.types import *

address_schema = StructType([
    StructField("City", StringType(), True),
    StructField("Pincode", IntegerType(), True)
])

employee_schema = StructType([
    StructField("EmployeeID", IntegerType(), False),
    StructField("Name", StringType(), True),
    StructField("Address", address_schema, True)
])

employee_df = spark.createDataFrame([
    (101, "Arun", ("Chennai", 600001)),
    (102, "Bala", ("Madurai", 625001))
], employee_schema)

employee_df.printSchema()
employee_df.show(truncate=False)

# Extract nested fields
employee_df.select(
    "Name",
    col("Address.City").alias("City"),
    col("Address.Pincode").alias("Pincode")
).show()


### An online shopping platform receives customer data from an API. Each customer record contains an address, multiple skills or interests, and flexible customer preferences.


A simplified `JSON record` could look like this:

{
  "CustomerID": 101,
  "Name": "Arun",
  "Skills": ["Python", "SQL", "PySpark"],
  "Preferences": {
    "Language": "English",
    "Notifications": "Enabled"
  },
  "Address": {
    "City": "Chennai",
    "Pincode": 600001
  }
}

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    ArrayType, MapType
)
from pyspark.sql.functions import explode, col

spark = SparkSession.builder.appName(
    "CustomerComplexTypes"
).getOrCreate()

# Define nested address structure
address_schema = StructType([
    StructField("City", StringType(), True),
    StructField("Pincode", IntegerType(), True)
])

# Define customer schema
customer_schema = StructType([
    StructField("CustomerID", IntegerType(), False),
    StructField("Name", StringType(), True),
    StructField(
        "Skills",
        ArrayType(StringType()),
        True
    ),
    StructField(
        "Preferences",
        MapType(StringType(), StringType()),
        True
    ),
    StructField("Address", address_schema, True)
])

# Create sample customer data
customer_data = [
    (
        101,
        "Arun",
        ["Python", "SQL", "PySpark"],
        {"Language": "English", "Notifications": "Enabled"},
        ("Chennai", 600001)
    ),
    (
        102,
        "Bala",
        ["Java", "SQL"],
        {"Language": "Tamil", "Notifications": "Disabled"},
        ("Madurai", 625001)
    )
]

customer_df = spark.createDataFrame(
    customer_data,
    schema=customer_schema
)

print("Original customer data:")
customer_df.show(truncate=False)

customer_df.printSchema()

### Interview Questions:


**Q1. What happens when explode() processes an array containing three elements?**

explode() is a PySpark function is used to `converts each element of an array into a separate row` in a DataFrame.

**Q2. What is the purpose of pivot()?**

- pivot() is used to `transforms distinct values from a column into separate columns`, allowing you to summarize data by categories.
- here we can use aggregation such as `sum(), count(), or avg()`.

**Q3. What is the difference between ArrayType and MapType?**

- ArrayType is used to `stores multiple values of the same data type in one column`.
- MapType is used to `stores key-value pairs in one column`.
- StructType is used to `stores name fields where one column contains different data types`.

**Q4. What is the difference between StructType and MapType?**

- StructType is used to `stores name fields where one column contains different data types`.
- MapType is used to `stores key-value pairs in one column`.

**Q5. Which complex type is useful for nested JSON data?**

- `ArrayType, MapType, and StructType` can all represent nested JSON structures.
- depending on the `data's structure`.